In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.cluster import DBSCAN
import utm
from shapely.geometry import Point
from shapely.ops import unary_union, snap
from shapely.errors import TopologicalError

# =============================================================================
# 🔥 兼容 .py 和 .ipynb（关键修复）
# =============================================================================
BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent

OUTPUT_DIR = BASE_DIR / "results" / "notebook-default"
OUTPUT_DIR.mkdir(exist_ok=True)

# =========================
# 海岸线文件路径
# =========================
COASTLINE_SHP = BASE_DIR / "data" / "geospatial" / "ne_10m_coastline" / "ne_10m_coastline.shp"
CHINA_SHP = BASE_DIR / "data" / "geospatial" / "ne_10m_admin_0_countries" / "ne_10m_admin_0_countries.shp"

# =========================
# 参数
# =========================
USE_COLS = [
    "MMSI","BaseDateTime","Latitude","Longitude","SOG","COG","Heading",
    "Status","ShipName","DataType","utc","datetime"
]
DTYPE_MAP = {"MMSI": "int64", "Latitude": "float64", "Longitude": "float64"}

STOP_SPEED_KN = 1
MIN_STOP_MINUTES = 60
BERTH_DRIFT_MAX = 150
ANCH_DRIFT_MIN = 300
ANCH_DURATION_MIN = 24 * 60

BERTH_EPS = 150
BERTH_MIN_SAMPLES = 2
ANCH_EPS = 6000
ANCH_MIN_SAMPLES = 5

BERTH_COAST_MAX = 4000
ANCH_COAST_MIN = 3000
ANCH_COAST_MAX = 20000

PORT_EPS = 1000
PORT_MIN_SAMPLES = 1

CHINA_LON_MIN = 73
CHINA_LON_MAX = 135
CHINA_LAT_MIN = 18
CHINA_LAT_MAX = 54

# =========================
# 工具函数
# =========================
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1-a))

def to_utm(lat, lon):
    return utm.from_latlon(lat, lon)[:2]

# =========================
# 海岸线
# =========================
def get_china_coastline_line():
    coast = gpd.read_file(COASTLINE_SHP)
    world = gpd.read_file(CHINA_SHP)
    china = world[world["NAME"].isin(["China", "Taiwan"])].unary_union
    china_coast = coast[coast.intersects(china)]
    return china_coast.to_crs(3857).unary_union

def add_real_coast_distance(df, coast_line_3857):
    print("🌊 计算到海岸线真实距离（点→线）")
    if len(df) == 0:
        return df
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326).to_crs(3857)
    gdf["distance_to_coast"] = gdf.geometry.distance(coast_line_3857)
    return pd.DataFrame(gdf.drop(columns="geometry"))

# =========================
# 泊位矩形
# =========================
def cluster_to_rectangle(df_clustered, output_type):
    results = []
    if df_clustered.empty:
        return pd.DataFrame()
    for cid, g in df_clustered.groupby("cluster"):
        if cid == -1: continue
        clat, clon = g.lat.mean(), g.lon.mean()
        min_lon, max_lon = g.lon.min(), g.lon.max()
        min_lat, max_lat = g.lat.min(), g.lat.max()
        poly = f"POLYGON(({min_lon} {min_lat}, {max_lon} {min_lat}, {max_lon} {max_lat}, {min_lon} {max_lat}, {min_lon} {min_lat}))"
        results.append({
            "cluster_id": cid, "lat": clat, "lon": clon, "count": len(g),
            "type": output_type, "polygon": poly
        })
    return pd.DataFrame(results)

# =========================
# 锚地凸包
# =========================
def cluster_to_convex_hull(df_clustered, output_type):
    results = []
    if df_clustered.empty:
        return pd.DataFrame()
    for cid, g in df_clustered.groupby("cluster"):
        if cid == -1: continue
        pts = [Point(lon, lat) for lon, lat in zip(g.lon, g.lat)]
        hull = unary_union(pts).convex_hull
        results.append({
            "cluster_id": cid, "lat": g.lat.mean(), "lon": g.lon.mean(),
            "count": len(g), "type": output_type, "polygon": hull.wkt
        })
    return pd.DataFrame(results)

# =============================================================================
# 🔥🔥🔥 【已修改：港口贴岸 + 不再是三角形】
# =============================================================================
def build_ports(berths, coast_line_3857):
    print("🏭 港口聚类（贴岸优化 + 消除三角形）")
    if berths.empty:
        return pd.DataFrame()

    gdf = gpd.GeoDataFrame(
        berths,
        geometry=gpd.GeoSeries.from_wkt(berths["polygon"]),
        crs="EPSG:4326"
    ).to_crs(epsg=3857)

    coords = np.array([[p.centroid.x, p.centroid.y] for p in gdf.geometry])
    db = DBSCAN(eps=PORT_EPS, min_samples=PORT_MIN_SAMPLES).fit(coords)
    gdf["port_id"] = db.labels_

    ports = []
    for pid in sorted(gdf["port_id"].unique()):
        if pid == -1:
            continue

        group = gdf[gdf["port_id"] == pid]
        merged = unary_union(group.geometry)

        # 🔥 贴海岸线（核心解决你的问题）
        merged = snap(merged, coast_line_3857, tolerance=250)
        # 🔥 外扩让形状更大更规整
        merged = merged.buffer(180, join_style=2)

        try:
            hull = merged.convex_hull
        except TopologicalError:
            hull = merged.envelope

        hull_4326 = gpd.GeoSeries([hull], crs=3857).to_crs(4326).iloc[0]

        ports.append({
            "port_id": pid,
            "lat": hull_4326.centroid.y,
            "lon": hull_4326.centroid.x,
            "berth_count": len(group),
            "polygon": hull_4326.wkt
        })
    return pd.DataFrame(ports)

# =========================
# 以下不变
# =========================
def read_ship_csv(path):
    for e in ["utf-8","gbk","latin1"]:
        try: return pd.read_csv(path, usecols=USE_COLS, dtype=DTYPE_MAP, encoding=e)
        except: continue
    raise RuntimeError(f"读取失败 {path}")

def load_all_ships_data():
    folder = BASE_DIR / "data" / "ais" / "cleaned" / "data"
    files = list(folder.glob("*.csv"))
    df_list = []
    with ThreadPoolExecutor(min(8, len(files))) as exe:
        fut = [exe.submit(read_ship_csv, f) for f in files]
        for i,f in enumerate(as_completed(fut),1):
            df_list.append(f.result())
            print(f"✔ {i}/{len(fut)}")
    df = pd.concat(df_list, ignore_index=True)
    print("AIS总量:", len(df))
    return df

def prepare_ship_data(df):
    df = df.rename(columns={"BaseDateTime":"timestamp","Latitude":"lat","Longitude":"lon"})
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df.sort_values(["MMSI","timestamp"])

def detect_stops(df):
    gb = df.groupby("MMSI")
    df["lat_prev"] = gb["lat"].shift(1)
    df["lon_prev"] = gb["lon"].shift(1)
    df["time_prev"] = gb["timestamp"].shift(1)
    df["time_diff"] = (df["timestamp"] - df["time_prev"]).dt.total_seconds()
    df["distance"] = haversine(df["lat"],df["lon"],df["lat_prev"],df["lon_prev"])
    df["speed_kn"] = (df["distance"] / df["time_diff"]) * 1.94384
    df["speed_kn"] = df["speed_kn"].replace([np.inf,-np.inf],0).fillna(0)
    df["stop"] = (df["speed_kn"] < STOP_SPEED_KN).astype(int)
    df["stop_group"] = gb["stop"].transform(lambda x: (x!=x.shift(1)).cumsum())

    stops = df[df["stop"]==1].groupby(["MMSI","stop_group"]).agg(
        lat=("lat","mean"), lon=("lon","mean"),
        start=("timestamp","min"), end=("timestamp","max"),
        lat_min=("lat","min"), lat_max=("lat","max"),
        lon_min=("lon","min"), lon_max=("lon","max")
    )
    stops["duration"] = (stops["end"] - stops["start"]).dt.total_seconds()/60
    stops["drift"] = haversine(stops["lat_min"],stops["lon_min"],stops["lat_max"],stops["lon_max"])
    stops = stops[stops["duration"]>=MIN_STOP_MINUTES].reset_index()
    print("有效停泊段:", len(stops))
    return stops

def classify_behavior(stops):
    stops = stops.copy()
    stops["behavior"] = "unknown"
    stops.loc[stops["drift"] < BERTH_DRIFT_MAX, "behavior"] = "berth_like"
    stops.loc[(stops["drift"]>=ANCH_DRIFT_MIN)&(stops["duration"]>=ANCH_DURATION_MIN), "behavior"] = "anchorage_like"
    return stops

def run_dbscan(df_input, eps, min_samples):
    if df_input.empty: return pd.DataFrame()
    dfin = df_input.copy()
    dfin[["x","y"]] = dfin.apply(lambda r: pd.Series(to_utm(r.lat, r.lon)), axis=1)
    dfin["cluster"] = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(dfin[["x","y"]])
    return dfin

def add_land_flag(df):
    if df.empty: return df
    world = gpd.read_file(CHINA_SHP)
    china = world[world["NAME"].isin(["China","Taiwan"])].to_crs(3857).unary_union
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326).to_crs(3857)
    gdf["inside_china_land"] = gdf.geometry.within(china)
    return pd.DataFrame(gdf.drop(columns="geometry"))

def filter_berths(berths):
    coastal = berths[berths["distance_to_coast"] <= BERTH_COAST_MAX]
    inland = berths[(berths["distance_to_coast"]>BERTH_COAST_MAX)&(berths["inside_china_land"]==True)]
    return pd.concat([coastal,inland], ignore_index=True)

def filter_anchorages(anchorages):
    return anchorages[(anchorages["distance_to_coast"]>=ANCH_COAST_MIN)&(anchorages["distance_to_coast"]<=ANCH_COAST_MAX)].copy()

def filter_china(df):
    if df.empty: return df
    return df[(df.lon>=73)&(df.lon<=135)&(df.lat>=18)&(df.lat<=54)].copy()

def export_csv(berths, anchorages, ports):
    berths.to_csv(OUTPUT_DIR/"china_berths.csv", index=False)
    anchorages.to_csv(OUTPUT_DIR/"china_anchorages.csv", index=False)
    ports.to_csv(OUTPUT_DIR/"china_ports.csv", index=False)
    print("✅ 导出完成")

# =========================
# 主程序
# =========================
def run():
    coast_line = get_china_coastline_line()
    df = load_all_ships_data()
    df = prepare_ship_data(df)
    stops = detect_stops(df)
    stops = classify_behavior(stops)

    b_cluster = run_dbscan(stops[stops.behavior=="berth_like"], BERTH_EPS, BERTH_MIN_SAMPLES)
    berths = cluster_to_rectangle(b_cluster, "berth")

    a_cluster = run_dbscan(stops[stops.behavior=="anchorage_like"], ANCH_EPS, ANCH_MIN_SAMPLES)
    anchorages = cluster_to_convex_hull(a_cluster, "anchorage")

    berths = add_real_coast_distance(berths, coast_line)
    anchorages = add_real_coast_distance(anchorages, coast_line)
    berths = add_land_flag(berths)

    berths = filter_berths(berths)
    anchorages = filter_anchorages(anchorages)

    # 🔥 传入海岸线，让港口贴岸
    ports = build_ports(berths, coast_line)

    berths = filter_china(berths)
    anchorages = filter_china(anchorages)
    ports = filter_china(ports)

    print("\n===== 最终结果 =====")
    print("泊位（矩形）：", len(berths))
    print("锚地（真实轮廓）：", len(anchorages))
    print("港口（已贴岸 + 无三角形）：", len(ports))

    export_csv(berths, anchorages, ports)

if __name__ == "__main__":
    run()

C:\Users\Aden\AppData\Local\Temp\ipykernel_6764\2180389344.py:81: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  china = world[world["NAME"].isin(["China", "Taiwan"])].unary_union
C:\Users\Aden\AppData\Local\Temp\ipykernel_6764\2180389344.py:83: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  return china_coast.to_crs(3857).unary_union


✔ 1/200
✔ 2/200
✔ 3/200
✔ 4/200
✔ 5/200
✔ 6/200
✔ 7/200
✔ 8/200
✔ 9/200
✔ 10/200
✔ 11/200
✔ 12/200
✔ 13/200
✔ 14/200
✔ 15/200
✔ 16/200
✔ 17/200
✔ 18/200
✔ 19/200
✔ 20/200
✔ 21/200
✔ 22/200
✔ 23/200
✔ 24/200
✔ 25/200
✔ 26/200
✔ 27/200
✔ 28/200
✔ 29/200
✔ 30/200
✔ 31/200
✔ 32/200
✔ 33/200
✔ 34/200
✔ 35/200
✔ 36/200
✔ 37/200
✔ 38/200
✔ 39/200
✔ 40/200
✔ 41/200
✔ 42/200
✔ 43/200
✔ 44/200
✔ 45/200
✔ 46/200
✔ 47/200
✔ 48/200
✔ 49/200
✔ 50/200
✔ 51/200
✔ 52/200
✔ 53/200
✔ 54/200
✔ 55/200
✔ 56/200
✔ 57/200
✔ 58/200
✔ 59/200
✔ 60/200
✔ 61/200
✔ 62/200
✔ 63/200
✔ 64/200
✔ 65/200
✔ 66/200
✔ 67/200
✔ 68/200
✔ 69/200
✔ 70/200
✔ 71/200
✔ 72/200
✔ 73/200
✔ 74/200
✔ 75/200
✔ 76/200
✔ 77/200
✔ 78/200
✔ 79/200
✔ 80/200
✔ 81/200
✔ 82/200
✔ 83/200
✔ 84/200
✔ 85/200
✔ 86/200
✔ 87/200
✔ 88/200
✔ 89/200
✔ 90/200
✔ 91/200
✔ 92/200
✔ 93/200
✔ 94/200
✔ 95/200
✔ 96/200
✔ 97/200
✔ 98/200
✔ 99/200
✔ 100/200
✔ 101/200
✔ 102/200
✔ 103/200
✔ 104/200
✔ 105/200
✔ 106/200
✔ 107/200
✔ 108/200
✔ 109/200
✔ 110/200
✔ 111/20

C:\Users\Aden\AppData\Local\Temp\ipykernel_6764\2180389344.py:244: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  china = world[world["NAME"].isin(["China","Taiwan"])].to_crs(3857).unary_union


🏭 港口聚类（贴岸优化 + 消除三角形）

===== 最终结果 =====
泊位（矩形）： 1348
锚地（真实轮廓）： 107
港口（已贴岸 + 无三角形）： 731
✅ 导出完成


输出到6.4

In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.cluster import DBSCAN
import utm
from shapely.geometry import Point
from shapely.ops import unary_union, snap
from shapely.errors import TopologicalError

# =============================================================================
# 🔥 输出文件夹：6.4 output
# =============================================================================
BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent

OUTPUT_DIR = BASE_DIR / "results" / "notebook-6.4"
OUTPUT_DIR.mkdir(exist_ok=True)

# =========================
# 海岸线文件路径
# =========================
COASTLINE_SHP = BASE_DIR / "data" / "geospatial" / "ne_10m_coastline" / "ne_10m_coastline.shp"
CHINA_SHP = BASE_DIR / "data" / "geospatial" / "ne_10m_admin_0_countries" / "ne_10m_admin_0_countries.shp"

# =========================
# 参数
# =========================
USE_COLS = [
    "MMSI","BaseDateTime","Latitude","Longitude","SOG","COG","Heading",
    "Status","ShipName","DataType","utc","datetime"
]
DTYPE_MAP = {"MMSI": "int64", "Latitude": "float64", "Longitude": "float64"}

STOP_SPEED_KN = 1
MIN_STOP_MINUTES = 60
BERTH_DRIFT_MAX = 150
ANCH_DRIFT_MIN = 300
ANCH_DURATION_MIN = 24 * 60

BERTH_EPS = 150
BERTH_MIN_SAMPLES = 2
ANCH_EPS = 6000
ANCH_MIN_SAMPLES = 5

BERTH_COAST_MAX = 4000
ANCH_COAST_MIN = 3000
ANCH_COAST_MAX = 20000

PORT_EPS = 1000
PORT_MIN_SAMPLES = 1

CHINA_LON_MIN = 73
CHINA_LON_MAX = 135
CHINA_LAT_MIN = 18
CHINA_LAT_MAX = 54

# =========================
# 工具函数
# =========================
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1-a))

def to_utm(lat, lon):
    return utm.from_latlon(lat, lon)[:2]

# =========================
# 海岸线
# =========================
def get_china_coastline_line():
    coast = gpd.read_file(COASTLINE_SHP)
    world = gpd.read_file(CHINA_SHP)
    china = world[world["NAME"].isin(["China", "Taiwan"])].unary_union
    china_coast = coast[coast.intersects(china)]
    return china_coast.to_crs(3857).unary_union

def add_real_coast_distance(df, coast_line_3857):
    print("🌊 计算到海岸线真实距离（点→线）")
    if len(df) == 0:
        return df
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326).to_crs(3857)
    gdf["distance_to_coast"] = gdf.geometry.distance(coast_line_3857)
    return pd.DataFrame(gdf.drop(columns="geometry"))

# =========================
# 泊位矩形
# =========================
def cluster_to_rectangle(df_clustered, output_type):
    results = []
    if df_clustered.empty:
        return pd.DataFrame()
    for cid, g in df_clustered.groupby("cluster"):
        if cid == -1: continue
        clat, clon = g.lat.mean(), g.lon.mean()
        min_lon, max_lon = g.lon.min(), g.lon.max()
        min_lat, max_lat = g.lat.min(), g.lat.max()
        poly = f"POLYGON(({min_lon} {min_lat}, {max_lon} {min_lat}, {max_lon} {max_lat}, {min_lon} {max_lat}, {min_lon} {min_lat}))"
        results.append({
            "cluster_id": cid, "lat": clat, "lon": clon, "count": len(g),
            "type": output_type, "polygon": poly
        })
    return pd.DataFrame(results)

# =========================
# 锚地凸包
# =========================
def cluster_to_convex_hull(df_clustered, output_type):
    results = []
    if df_clustered.empty:
        return pd.DataFrame()
    for cid, g in df_clustered.groupby("cluster"):
        if cid == -1: continue
        pts = [Point(lon, lat) for lon, lat in zip(g.lon, g.lat)]
        hull = unary_union(pts).convex_hull
        results.append({
            "cluster_id": cid, "lat": g.lat.mean(), "lon": g.lon.mean(),
            "count": len(g), "type": output_type, "polygon": hull.wkt
        })
    return pd.DataFrame(results)

# =============================================================================
# 港口聚类
# =============================================================================
def build_ports(berths, coast_line_3857):
    print("🏭 港口聚类（贴岸优化）")
    if berths.empty:
        return pd.DataFrame()

    gdf = gpd.GeoDataFrame(
        berths,
        geometry=gpd.GeoSeries.from_wkt(berths["polygon"]),
        crs="EPSG:4326"
    ).to_crs(epsg=3857)

    coords = np.array([[p.centroid.x, p.centroid.y] for p in gdf.geometry])
    db = DBSCAN(eps=PORT_EPS, min_samples=PORT_MIN_SAMPLES).fit(coords)
    gdf["port_id"] = db.labels_

    ports = []
    for pid in sorted(gdf["port_id"].unique()):
        if pid == -1:
            continue

        group = gdf[gdf["port_id"] == pid]
        merged = unary_union(group.geometry)
        merged = snap(merged, coast_line_3857, tolerance=250)
        merged = merged.buffer(180, join_style=2)

        try:
            hull = merged.convex_hull
        except TopologicalError:
            hull = merged.envelope

        hull_4326 = gpd.GeoSeries([hull], crs=3857).to_crs(4326).iloc[0]

        ports.append({
            "port_id": pid,
            "lat": hull_4326.centroid.y,
            "lon": hull_4326.centroid.x,
            "berth_count": len(group),
            "polygon": hull_4326.wkt
        })
    return pd.DataFrame(ports)

# =========================
# 数据读取
# =========================
def read_ship_csv(path):
    for e in ["utf-8","gbk","latin1"]:
        try: return pd.read_csv(path, usecols=USE_COLS, dtype=DTYPE_MAP, encoding=e)
        except: continue
    raise RuntimeError(f"读取失败 {path}")

def load_all_ships_data():
    folder = BASE_DIR / "data" / "ais" / "cleaned" / "data"
    files = list(folder.glob("*.csv"))
    df_list = []
    with ThreadPoolExecutor(min(8, len(files))) as exe:
        fut = [exe.submit(read_ship_csv, f) for f in files]
        for i,f in enumerate(as_completed(fut),1):
            df_list.append(f.result())
            print(f"✔ {i}/{len(fut)}")
    df = pd.concat(df_list, ignore_index=True)
    print("AIS总量:", len(df))
    return df

def prepare_ship_data(df):
    df = df.rename(columns={"BaseDateTime":"timestamp","Latitude":"lat","Longitude":"lon"})
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df.sort_values(["MMSI","timestamp"])

# =========================
# 停泊检测
# =========================
def detect_stops(df):
    gb = df.groupby("MMSI")
    df["lat_prev"] = gb["lat"].shift(1)
    df["lon_prev"] = gb["lon"].shift(1)
    df["time_prev"] = gb["timestamp"].shift(1)
    df["time_diff"] = (df["timestamp"] - df["time_prev"]).dt.total_seconds()
    df["distance"] = haversine(df["lat"],df["lon"],df["lat_prev"],df["lon_prev"])
    df["speed_kn"] = (df["distance"] / df["time_diff"]) * 1.94384
    df["speed_kn"] = df["speed_kn"].replace([np.inf,-np.inf],0).fillna(0)
    df["stop"] = (df["speed_kn"] < STOP_SPEED_KN).astype(int)
    df["stop_group"] = gb["stop"].transform(lambda x: (x!=x.shift(1)).cumsum())

    stops = df[df["stop"]==1].groupby(["MMSI","stop_group"]).agg(
        lat=("lat","mean"), lon=("lon","mean"),
        start=("timestamp","min"), end=("timestamp","max"),
        lat_min=("lat","min"), lat_max=("lat","max"),
        lon_min=("lon","min"), lon_max=("lon","max")
    )
    stops["duration"] = (stops["end"] - stops["start"]).dt.total_seconds()/60
    stops["drift"] = haversine(stops["lat_min"],stops["lon_min"],stops["lat_max"],stops["lon_max"])
    stops = stops[stops["duration"]>=MIN_STOP_MINUTES].reset_index()
    print("有效停泊段:", len(stops))
    return stops

# =========================
# 行为分类
# =========================
def classify_behavior(stops):
    stops = stops.copy()
    stops["behavior"] = "unknown"
    stops.loc[stops["drift"] < BERTH_DRIFT_MAX, "behavior"] = "berth_like"
    stops.loc[(stops["drift"]>=ANCH_DRIFT_MIN)&(stops["duration"]>=ANCH_DURATION_MIN), "behavior"] = "anchorage_like"
    return stops

# =========================
# 聚类
# =========================
def run_dbscan(df_input, eps, min_samples):
    if df_input.empty: return pd.DataFrame()
    dfin = df_input.copy()
    dfin[["x","y"]] = dfin.apply(lambda r: pd.Series(to_utm(r.lat, r.lon)), axis=1)
    dfin["cluster"] = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(dfin[["x","y"]])
    return dfin

def add_land_flag(df):
    if df.empty: return df
    world = gpd.read_file(CHINA_SHP)
    china = world[world["NAME"].isin(["China","Taiwan"])].to_crs(3857).unary_union
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326).to_crs(3857)
    gdf["inside_china_land"] = gdf.geometry.within(china)
    return pd.DataFrame(gdf.drop(columns="geometry"))

# =========================
# 过滤
# =========================
def filter_berths(berths):
    coastal = berths[berths["distance_to_coast"] <= BERTH_COAST_MAX]
    inland = berths[(berths["distance_to_coast"]>BERTH_COAST_MAX)&(berths["inside_china_land"]==True)]
    return pd.concat([coastal,inland], ignore_index=True)

# 🔥 🔥 🔥 核心：只保留海岸线右侧（海洋）锚地，剔除左侧（陆地/内河）
def filter_anchorages(anchorages):
    df = anchorages[
        (anchorages["distance_to_coast"] >= ANCH_COAST_MIN) &
        (anchorages["distance_to_coast"] <= ANCH_COAST_MAX)
    ].copy()

    if df.empty:
        return df

    coast_3857 = get_china_coastline_line()
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326).to_crs(3857)

    def is_land_side(pt, line):
        try:
            proj = line.project(pt)
            cp = line.interpolate(proj)
            dx = pt.x - cp.x
            return dx < 0
        except:
            return True

    gdf["is_land"] = gdf.geometry.apply(lambda p: is_land_side(p, coast_3857))
    df = df[~gdf["is_land"]].copy()
    return df

def filter_china(df):
    if df.empty: return df
    return df[(df.lon>=73)&(df.lon<=135)&(df.lat>=18)&(df.lat<=54)].copy()

# =========================
# 导出到 6.4 output
# =========================
def export_csv(berths, anchorages, ports):
    berths.to_csv(OUTPUT_DIR / "china_berths.csv", index=False)
    anchorages.to_csv(OUTPUT_DIR / "china_anchorages.csv", index=False)
    ports.to_csv(OUTPUT_DIR / "china_ports.csv", index=False)
    print(f"✅ 全部导出到：{OUTPUT_DIR}")

# =========================
# 主程序
# =========================
def run():
    coast_line = get_china_coastline_line()
    df = load_all_ships_data()
    df = prepare_ship_data(df)
    stops = detect_stops(df)
    stops = classify_behavior(stops)

    b_cluster = run_dbscan(stops[stops.behavior=="berth_like"], BERTH_EPS, BERTH_MIN_SAMPLES)
    berths = cluster_to_rectangle(b_cluster, "berth")

    a_cluster = run_dbscan(stops[stops.behavior=="anchorage_like"], ANCH_EPS, ANCH_MIN_SAMPLES)
    anchorages = cluster_to_convex_hull(a_cluster, "anchorage")

    berths = add_real_coast_distance(berths, coast_line)
    anchorages = add_real_coast_distance(anchorages, coast_line)
    berths = add_land_flag(berths)

    berths = filter_berths(berths)
    anchorages = filter_anchorages(anchorages)

    ports = build_ports(berths, coast_line)

    berths = filter_china(berths)
    anchorages = filter_china(anchorages)
    ports = filter_china(ports)

    print("\n===== 最终结果 =====")
    print("泊位：", len(berths))
    print("锚地（仅海洋侧）：", len(anchorages))
    print("港口：", len(ports))

    export_csv(berths, anchorages, ports)

if __name__ == "__main__":
    run()

C:\Users\Aden\AppData\Local\Temp\ipykernel_6764\1831030656.py:81: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  china = world[world["NAME"].isin(["China", "Taiwan"])].unary_union
C:\Users\Aden\AppData\Local\Temp\ipykernel_6764\1831030656.py:83: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  return china_coast.to_crs(3857).unary_union


✔ 1/200
✔ 2/200
✔ 3/200
✔ 4/200
✔ 5/200
✔ 6/200
✔ 7/200
✔ 8/200
✔ 9/200
✔ 10/200
✔ 11/200
✔ 12/200
✔ 13/200
✔ 14/200
✔ 15/200
✔ 16/200
✔ 17/200
✔ 18/200
✔ 19/200
✔ 20/200
✔ 21/200
✔ 22/200
✔ 23/200
✔ 24/200
✔ 25/200
✔ 26/200
✔ 27/200
✔ 28/200
✔ 29/200
✔ 30/200
✔ 31/200
✔ 32/200
✔ 33/200
✔ 34/200
✔ 35/200
✔ 36/200
✔ 37/200
✔ 38/200
✔ 39/200
✔ 40/200
✔ 41/200
✔ 42/200
✔ 43/200
✔ 44/200
✔ 45/200
✔ 46/200
✔ 47/200
✔ 48/200
✔ 49/200
✔ 50/200
✔ 51/200
✔ 52/200
✔ 53/200
✔ 54/200
✔ 55/200
✔ 56/200
✔ 57/200
✔ 58/200
✔ 59/200
✔ 60/200
✔ 61/200
✔ 62/200
✔ 63/200
✔ 64/200
✔ 65/200
✔ 66/200
✔ 67/200
✔ 68/200
✔ 69/200
✔ 70/200
✔ 71/200
✔ 72/200
✔ 73/200
✔ 74/200
✔ 75/200
✔ 76/200
✔ 77/200
✔ 78/200
✔ 79/200
✔ 80/200
✔ 81/200
✔ 82/200
✔ 83/200
✔ 84/200
✔ 85/200
✔ 86/200
✔ 87/200
✔ 88/200
✔ 89/200
✔ 90/200
✔ 91/200
✔ 92/200
✔ 93/200
✔ 94/200
✔ 95/200
✔ 96/200
✔ 97/200
✔ 98/200
✔ 99/200
✔ 100/200
✔ 101/200
✔ 102/200
✔ 103/200
✔ 104/200
✔ 105/200
✔ 106/200
✔ 107/200
✔ 108/200
✔ 109/200
✔ 110/200
✔ 111/20

C:\Users\Aden\AppData\Local\Temp\ipykernel_6764\1831030656.py:250: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  china = world[world["NAME"].isin(["China","Taiwan"])].to_crs(3857).unary_union
C:\Users\Aden\AppData\Local\Temp\ipykernel_6764\1831030656.py:81: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  china = world[world["NAME"].isin(["China", "Taiwan"])].unary_union
C:\Users\Aden\AppData\Local\Temp\ipykernel_6764\1831030656.py:83: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  return china_coast.to_crs(3857).unary_union


🏭 港口聚类（贴岸优化）

===== 最终结果 =====
泊位： 1348
锚地（仅海洋侧）： 71
港口： 731
✅ 全部导出到：d:\code\聚类\6.4 output


尝试修改“输出到6.4”（还需要改锚地太少了

In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.cluster import DBSCAN
import utm
from shapely.geometry import Point
from shapely.ops import unary_union, snap
from shapely.errors import TopologicalError

# =============================================================================
# 🔥 输出文件夹：7.20
# =============================================================================
BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent

OUTPUT_DIR = BASE_DIR / "results" / "notebook-7.20"
OUTPUT_DIR.mkdir(exist_ok=True)

# =========================
# 海岸线文件路径
# =========================
COASTLINE_SHP = BASE_DIR / "data" / "geospatial" / "ne_10m_coastline" / "ne_10m_coastline.shp"
CHINA_SHP = BASE_DIR / "data" / "geospatial" / "ne_10m_admin_0_countries" / "ne_10m_admin_0_countries.shp"

# =========================
# 参数配置
# =========================
USE_COLS = [
    "MMSI","BaseDateTime","Latitude","Longitude","SOG","COG","Heading",
    "Status","ShipName","DataType","utc","datetime"
]
DTYPE_MAP = {"MMSI": "int64", "Latitude": "float64", "Longitude": "float64"}

STOP_SPEED_KN = 1
MIN_STOP_MINUTES = 60
BERTH_DRIFT_MAX = 150
# 放宽锚泊判定，增加锚地识别数量
ANCH_DRIFT_MIN = 200
ANCH_DURATION_MIN = 12 * 60

BERTH_EPS = 150
BERTH_MIN_SAMPLES = 2
# 锚地聚类放宽
ANCH_EPS = 8000
ANCH_MIN_SAMPLES = 3

BERTH_COAST_MAX = 4000
ANCH_COAST_MIN = 3000
ANCH_COAST_MAX = 20000

# 港口聚类半径维持1000不变，识别位置保持原样
PORT_EPS = 1000
PORT_MIN_SAMPLES = 1

CHINA_LON_MIN = 73
CHINA_LON_MAX = 135
CHINA_LAT_MIN = 18
CHINA_LAT_MAX = 54

# =========================
# 工具函数
# =========================
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1-a))

def to_utm(lat, lon):
    return utm.from_latlon(lat, lon)[:2]

# =========================
# 海岸线
# =========================
def get_china_coastline_line():
    coast = gpd.read_file(COASTLINE_SHP)
    world = gpd.read_file(CHINA_SHP)
    china = world[world["NAME"].isin(["China", "Taiwan"])].unary_union
    china_coast = coast[coast.intersects(china)]
    return china_coast.to_crs(3857).unary_union

def add_real_coast_distance(df, coast_line_3857):
    print("🌊 计算到海岸线真实距离（点→线）")
    if len(df) == 0:
        return df
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326).to_crs(3857)
    gdf["distance_to_coast"] = gdf.geometry.distance(coast_line_3857)
    return pd.DataFrame(gdf.drop(columns="geometry"))

# =========================
# 泊位矩形
# =========================
def cluster_to_rectangle(df_clustered, output_type):
    results = []
    if df_clustered.empty:
        return pd.DataFrame()
    for cid, g in df_clustered.groupby("cluster"):
        if cid == -1: continue
        clat, clon = g.lat.mean(), g.lon.mean()
        min_lon, max_lon = g.lon.min(), g.lon.max()
        min_lat, max_lat = g.lat.min(), g.lat.max()
        poly = f"POLYGON(({min_lon} {min_lat}, {max_lon} {min_lat}, {max_lon} {max_lat}, {min_lon} {max_lat}, {min_lon} {min_lat}))"
        results.append({
            "cluster_id": cid, "lat": clat, "lon": clon, "count": len(g),
            "type": output_type, "polygon": poly
        })
    return pd.DataFrame(results)

# =========================
# 锚地凸包：固定300米缓冲，保证面积大于港口
# =========================
def cluster_to_convex_hull(df_clustered, output_type):
    results = []
    if df_clustered.empty:
        return pd.DataFrame()
    gdf = gpd.GeoDataFrame(df_clustered, geometry=gpd.points_from_xy(df_clustered.lon, df_clustered.lat), crs=4326).to_crs(3857)
    for cid, g in gdf.groupby("cluster"):
        if cid == -1: continue
        union_pts = unary_union(g.geometry)
        buffered = union_pts.buffer(300, cap_style=2, join_style=2)
        hull = buffered.convex_hull
        hull_wgs = gpd.GeoSeries([hull], crs=3857).to_crs(4326).iloc[0]
        orig_group = df_clustered[df_clustered.cluster == cid]
        results.append({
            "cluster_id": cid,
            "lat": orig_group.lat.mean(),
            "lon": orig_group.lon.mean(),
            "count": len(g),
            "type": output_type,
            "polygon": hull_wgs.wkt
        })
    return pd.DataFrame(results)

# =============================================================================
# 港口函数：聚类范围不变，仅加大buffer放大面积，中心位置不变
# =============================================================================
def build_ports(berths, coast_line_3857):
    print("🏭 港口聚类（点位不变，放大多边形面积）")
    if berths.empty:
        return pd.DataFrame()

    gdf = gpd.GeoDataFrame(
        berths,
        geometry=gpd.GeoSeries.from_wkt(berths["polygon"]),
        crs="EPSG:4326"
    ).to_crs(epsg=3857)

    coords = np.array([[p.centroid.x, p.centroid.y] for p in gdf.geometry])
    # PORT_EPS保持1000，港口识别位置完全不变
    db = DBSCAN(eps=PORT_EPS, min_samples=PORT_MIN_SAMPLES).fit(coords)
    gdf["port_id"] = db.labels_

    ports = []
    for pid in sorted(gdf["port_id"].unique()):
        if pid == -1:
            continue

        group = gdf[gdf["port_id"] == pid]
        merged = unary_union(group.geometry)
        merged = snap(merged, coast_line_3857, tolerance=250)
        # 缓冲距离500米，单纯放大港口多边形，不改变中心点位
        merged = merged.buffer(500, join_style=2)

        try:
            hull = merged.convex_hull
        except TopologicalError:
            hull = merged.envelope

        hull_4326 = gpd.GeoSeries([hull], crs=3857).to_crs(4326).iloc[0]

        ports.append({
            "port_id": pid,
            "lat": hull_4326.centroid.y,
            "lon": hull_4326.centroid.x,
            "berth_count": len(group),
            "polygon": hull_4326.wkt
        })
    return pd.DataFrame(ports)

# =========================
# 数据读取
# =========================
def read_ship_csv(path):
    for e in ["utf-8","gbk","latin1"]:
        try: return pd.read_csv(path, usecols=USE_COLS, dtype=DTYPE_MAP, encoding=e)
        except: continue
    raise RuntimeError(f"读取失败 {path}")

def load_all_ships_data():
    folder = BASE_DIR / "data" / "ais" / "cleaned" / "data"
    files = list(folder.glob("*.csv"))
    df_list = []
    with ThreadPoolExecutor(min(8, len(files))) as exe:
        fut = [exe.submit(read_ship_csv, f) for f in files]
        for i,f in enumerate(as_completed(fut),1):
            df_list.append(f.result())
            print(f"✔ {i}/{len(fut)}")
    df = pd.concat(df_list, ignore_index=True)
    print("AIS总量:", len(df))
    return df

def prepare_ship_data(df):
    df = df.rename(columns={"BaseDateTime":"timestamp","Latitude":"lat","Longitude":"lon"})
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df.sort_values(["MMSI","timestamp"])

# =========================
# 停泊检测
# =========================
def detect_stops(df):
    gb = df.groupby("MMSI")
    df["lat_prev"] = gb["lat"].shift(1)
    df["lon_prev"] = gb["lon"].shift(1)
    df["time_prev"] = gb["timestamp"].shift(1)
    df["time_diff"] = (df["timestamp"] - df["time_prev"]).dt.total_seconds()
    df["distance"] = haversine(df["lat"],df["lon"],df["lat_prev"],df["lon_prev"])
    df["speed_kn"] = (df["distance"] / df["time_diff"]) * 1.94384
    df["speed_kn"] = df["speed_kn"].replace([np.inf,-np.inf],0).fillna(0)
    df["stop"] = (df["speed_kn"] < STOP_SPEED_KN).astype(int)
    df["stop_group"] = gb["stop"].transform(lambda x: (x!=x.shift(1)).cumsum())

    stops = df[df["stop"]==1].groupby(["MMSI","stop_group"]).agg(
        lat=("lat","mean"), lon=("lon","mean"),
        start=("timestamp","min"), end=("timestamp","max"),
        lat_min=("lat","min"), lat_max=("lat","max"),
        lon_min=("lon","min"), lon_max=("lon","max")
    )
    stops["duration"] = (stops["end"] - stops["start"]).dt.total_seconds()/60
    stops["drift"] = haversine(stops["lat_min"],stops["lon_min"],stops["lat_max"],stops["lon_max"])
    stops = stops[stops["duration"]>=MIN_STOP_MINUTES].reset_index()
    print("有效停泊段:", len(stops))
    return stops

# =========================
# 行为分类
# =========================
def classify_behavior(stops):
    stops = stops.copy()
    stops["behavior"] = "unknown"
    stops.loc[stops["drift"] < BERTH_DRIFT_MAX, "behavior"] = "berth_like"
    stops.loc[(stops["drift"]>=ANCH_DRIFT_MIN)&(stops["duration"]>=ANCH_DURATION_MIN), "behavior"] = "anchorage_like"
    return stops

# =========================
# 聚类
# =========================
def run_dbscan(df_input, eps, min_samples):
    if df_input.empty: return pd.DataFrame()
    dfin = df_input.copy()
    dfin[["x","y"]] = dfin.apply(lambda r: pd.Series(to_utm(r.lat, r.lon)), axis=1)
    dfin["cluster"] = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(dfin[["x","y"]])
    return dfin

def add_land_flag(df):
    if df.empty: return df
    world = gpd.read_file(CHINA_SHP)
    china = world[world["NAME"].isin(["China","Taiwan"])].to_crs(3857).unary_union
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326).to_crs(3857)
    gdf["inside_china_land"] = gdf.geometry.within(china)
    return pd.DataFrame(gdf.drop(columns="geometry"))

# =========================
# 过滤
# =========================
def filter_berths(berths):
    coastal = berths[berths["distance_to_coast"] <= BERTH_COAST_MAX]
    inland = berths[(berths["distance_to_coast"]>BERTH_COAST_MAX)&(berths["inside_china_land"]==True)]
    return pd.concat([coastal,inland], ignore_index=True)

def filter_anchorages(anchorages):
    df = anchorages[
        (anchorages["distance_to_coast"] >= ANCH_COAST_MIN) &
        (anchorages["distance_to_coast"] <= ANCH_COAST_MAX)
    ].copy()

    if df.empty:
        return df

    coast_3857 = get_china_coastline_line()
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs=4326).to_crs(3857)

    def is_land_side(pt, line):
        try:
            proj = line.project(pt)
            cp = line.interpolate(proj)
            dx = pt.x - cp.x
            return dx < 0
        except:
            return True

    gdf["is_land"] = gdf.geometry.apply(lambda p: is_land_side(p, coast_3857))
    df = df[~gdf["is_land"]].copy()
    return df

def filter_china(df):
    if df.empty: return df
    return df[(df.lon>=73)&(df.lon<=135)&(df.lat>=18)&(df.lat<=54)].copy()

# =========================
# 导出到 7.20 文件夹
# =========================
def export_csv(berths, anchorages, ports):
    berths.to_csv(OUTPUT_DIR / "china_berths.csv", index=False)
    anchorages.to_csv(OUTPUT_DIR / "china_anchorages.csv", index=False)
    ports.to_csv(OUTPUT_DIR / "china_ports.csv", index=False)
    print(f"✅ 全部导出到：{OUTPUT_DIR}")

# =========================
# 主程序
# =========================
def run():
    coast_line = get_china_coastline_line()
    df = load_all_ships_data()
    df = prepare_ship_data(df)
    stops = detect_stops(df)
    stops = classify_behavior(stops)

    b_cluster = run_dbscan(stops[stops.behavior=="berth_like"], BERTH_EPS, BERTH_MIN_SAMPLES)
    berths = cluster_to_rectangle(b_cluster, "berth")

    a_cluster = run_dbscan(stops[stops.behavior=="anchorage_like"], ANCH_EPS, ANCH_MIN_SAMPLES)
    anchorages = cluster_to_convex_hull(a_cluster, "anchorage")

    berths = add_real_coast_distance(berths, coast_line)
    anchorages = add_real_coast_distance(anchorages, coast_line)
    berths = add_land_flag(berths)

    berths = filter_berths(berths)
    anchorages = filter_anchorages(anchorages)

    ports = build_ports(berths, coast_line)

    berths = filter_china(berths)
    anchorages = filter_china(anchorages)
    ports = filter_china(ports)

    print("\n===== 最终结果 =====")
    print("泊位：", len(berths))
    print("锚地（仅海洋侧）：", len(anchorages))
    print("港口：", len(ports))

    export_csv(berths, anchorages, ports)

if __name__ == "__main__":
    run()


C:\Users\Aden\AppData\Local\Temp\ipykernel_6764\2406354692.py:84: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  china = world[world["NAME"].isin(["China", "Taiwan"])].unary_union
C:\Users\Aden\AppData\Local\Temp\ipykernel_6764\2406354692.py:86: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  return china_coast.to_crs(3857).unary_union


✔ 1/200
✔ 2/200
✔ 3/200
✔ 4/200
✔ 5/200
✔ 6/200
✔ 7/200
✔ 8/200
✔ 9/200
✔ 10/200
✔ 11/200
✔ 12/200
✔ 13/200
✔ 14/200
✔ 15/200
✔ 16/200
✔ 17/200
✔ 18/200
✔ 19/200
✔ 20/200
✔ 21/200
✔ 22/200
✔ 23/200
✔ 24/200
✔ 25/200
✔ 26/200
✔ 27/200
✔ 28/200
✔ 29/200
✔ 30/200
✔ 31/200
✔ 32/200
✔ 33/200
✔ 34/200
✔ 35/200
✔ 36/200
✔ 37/200
✔ 38/200
✔ 39/200
✔ 40/200
✔ 41/200
✔ 42/200
✔ 43/200
✔ 44/200
✔ 45/200
✔ 46/200
✔ 47/200
✔ 48/200
✔ 49/200
✔ 50/200
✔ 51/200
✔ 52/200
✔ 53/200
✔ 54/200
✔ 55/200
✔ 56/200
✔ 57/200
✔ 58/200
✔ 59/200
✔ 60/200
✔ 61/200
✔ 62/200
✔ 63/200
✔ 64/200
✔ 65/200
✔ 66/200
✔ 67/200
✔ 68/200
✔ 69/200
✔ 70/200
✔ 71/200
✔ 72/200
✔ 73/200
✔ 74/200
✔ 75/200
✔ 76/200
✔ 77/200
✔ 78/200
✔ 79/200
✔ 80/200
✔ 81/200
✔ 82/200
✔ 83/200
✔ 84/200
✔ 85/200
✔ 86/200
✔ 87/200
✔ 88/200
✔ 89/200
✔ 90/200
✔ 91/200
✔ 92/200
✔ 93/200
✔ 94/200
✔ 95/200
✔ 96/200
✔ 97/200
✔ 98/200
✔ 99/200
✔ 100/200
✔ 101/200
✔ 102/200
✔ 103/200
✔ 104/200
✔ 105/200
✔ 106/200
✔ 107/200
✔ 108/200
✔ 109/200
✔ 110/200
✔ 111/20

C:\Users\Aden\AppData\Local\Temp\ipykernel_6764\2406354692.py:263: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  china = world[world["NAME"].isin(["China","Taiwan"])].to_crs(3857).unary_union
C:\Users\Aden\AppData\Local\Temp\ipykernel_6764\2406354692.py:84: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  china = world[world["NAME"].isin(["China", "Taiwan"])].unary_union
C:\Users\Aden\AppData\Local\Temp\ipykernel_6764\2406354692.py:86: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  return china_coast.to_crs(3857).unary_union


🏭 港口聚类（点位不变，放大多边形面积）

===== 最终结果 =====
泊位： 1348
锚地（仅海洋侧）： 63
港口： 731
✅ 全部导出到：d:\code\聚类\7.20


实际数据中国版

In [3]:
from pathlib import Path
import pandas as pd
from shapely import wkt

# ===================== 配置 =====================
# 输入文件路径
BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent
FILE_DOCK = BASE_DIR / "data" / "reference" / "实际数据" / "码头.csv"
FILE_ANCH = BASE_DIR / "data" / "reference" / "实际数据" / "锚地.csv"
# 输出文件夹
OUT_DIR = BASE_DIR / "results" / "reference-china-filter"
OUT_DIR.mkdir(exist_ok=True)

# 中国经纬度边界（和你原有代码统一）
LON_MIN, LON_MAX = 73, 135
LAT_MIN, LAT_MAX = 18, 54

# 兼容多编码读取csv
def read_csv_safe(file_path: Path):
    enc_list = ["utf-8", "gbk", "latin1", "utf-8-sig"]
    for enc in enc_list:
        try:
            return pd.read_csv(file_path, encoding=enc)
        except Exception:
            continue
    raise RuntimeError(f"读取失败：{file_path}")

# 从WKT geom提取中心点经纬度，再过滤中国范围
def filter_china_area(df: pd.DataFrame) -> pd.DataFrame:
    if "geom" not in df.columns:
        raise Exception("表格缺少geom坐标字段，无法提取经纬度")
    
    def get_center_lonlat(wkt_str):
        try:
            geo = wkt.loads(wkt_str)
            # 获取几何中心点
            cent = geo.centroid
            return cent.x, cent.y
        except:
            return None, None

    # 批量提取中心经纬度
    df[["tmp_lon", "tmp_lat"]] = df["geom"].apply(lambda x: pd.Series(get_center_lonlat(x)))
    # 剔除解析失败空值
    df = df.dropna(subset=["tmp_lon", "tmp_lat"])
    # 中国范围掩码
    mask = (df["tmp_lon"] >= LON_MIN) & (df["tmp_lon"] <= LON_MAX) & \
           (df["tmp_lat"] >= LAT_MIN) & (df["tmp_lat"] <= LAT_MAX)
    df_china = df[mask].copy()
    # 删除临时坐标列，还原原始字段
    df_china = df_china.drop(columns=["tmp_lon", "tmp_lat"])
    return df_china

# 处理单个文件并保存
def process_one_file(in_path: Path, out_folder: Path):
    print(f"\n正在处理：{in_path.name}")
    df_raw = read_csv_safe(in_path)
    total_raw = len(df_raw)
    df_china = filter_china_area(df_raw)
    total_china = len(df_china)

    save_path = out_folder / in_path.name
    df_china.to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"原始数据量：{total_raw} 条")
    print(f"中国范围保留：{total_china} 条")
    print(f"已保存至：{save_path.resolve()}")

if __name__ == "__main__":
    process_one_file(FILE_DOCK, OUT_DIR)
    process_one_file(FILE_ANCH, OUT_DIR)
    print("\n===== 全部处理完成 =====")
    print(f"过滤后文件存放目录：{OUT_DIR.resolve()}")



正在处理：码头.csv
原始数据量：1029 条
中国范围保留：931 条
已保存至：D:\code\聚类\实际数据\实际数据中国版\码头.csv

正在处理：锚地.csv
原始数据量：156 条
中国范围保留：156 条
已保存至：D:\code\聚类\实际数据\实际数据中国版\锚地.csv

===== 全部处理完成 =====
过滤后文件存放目录：D:\code\聚类\实际数据\实际数据中国版
